# Practical 8: Distributed Data Processing with PySpark DataFrames

**Name:** Vedant Vinod Rakshe

**Aim:** To use PySpark to read a CSV dataset and display its schema, perform filtering, grouping and aggregation, remove duplicate records, join two datasets and calculate the average sales by product category.

Run this cell once if PySpark is not installed (Google Colab already has Java).

In [ ]:
%pip install -q pyspark

## Task 1: Create Spark Session and Input Datasets

In [1]:
# Name: Vedant Vinod Rakshe
print("Name: Vedant Vinod Rakshe")
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum as _sum, count, max as _max, round as _round, desc

spark = (SparkSession.builder.appName("Practical_8")
         .master("local[2]").config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

# Create the two input datasets as CSV files
with open("sales_data.csv", "w") as f:
    f.write("order_id,customer_id,product,category,sales,quantity,region\n")
    rows = [
        "1,C01,Laptop,Electronics,55000,1,West", "2,C02,Mouse,Electronics,500,2,West",
        "3,C03,Chair,Furniture,3500,2,North",    "4,C01,Keyboard,Electronics,1500,1,West",
        "5,C04,Notebook,Stationery,300,5,South", "6,C05,Desk,Furniture,9000,1,North",
        "7,C02,Pen Pack,Stationery,120,3,West",  "8,C06,Monitor,Electronics,12000,1,East",
        "3,C03,Chair,Furniture,3500,2,North",    "9,C07,Sofa,Furniture,25000,1,South",
        "5,C04,Notebook,Stationery,300,5,South", "10,C09,Pencil Box,Stationery,80,4,East",
    ]
    f.write("\n".join(rows) + "\n")
with open("customers.csv", "w") as f:
    f.write("customer_id,customer_name,city\n")
    f.write("C01,Amit Shah,Mumbai\nC02,Riya Patil,Pune\nC03,Neha Rao,Delhi\n")
    f.write("C04,Karan Jain,Nashik\nC05,Sana Khan,Pune\nC06,Dev Mehta,Surat\nC07,Tina Dsouza,Goa\n")

Name: Vedant Vinod Rakshe
Spark version: 4.2.0


## Task 2: Read a CSV Dataset and Display its Schema

In [2]:
# Read a CSV dataset and display its schema
print("Name: Vedant Vinod Rakshe")
df_sales = spark.read.csv("sales_data.csv", header=True, inferSchema=True)
df_customers = spark.read.csv("customers.csv", header=True, inferSchema=True)

df_sales.printSchema()
print("Total rows:", df_sales.count())
df_sales.show()

Name: Vedant Vinod Rakshe
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)

Total rows: 12
+--------+-----------+----------+-----------+-----+--------+------+
|order_id|customer_id|   product|   category|sales|quantity|region|
+--------+-----------+----------+-----------+-----+--------+------+
|       1|        C01|    Laptop|Electronics|55000|       1|  West|
|       2|        C02|     Mouse|Electronics|  500|       2|  West|
|       3|        C03|     Chair|  Furniture| 3500|       2| North|
|       4|        C01|  Keyboard|Electronics| 1500|       1|  West|
|       5|        C04|  Notebook| Stationery|  300|       5| South|
|       6|        C05|      Desk|  Furniture| 9000|       1| North|
|       7|        C02|  Pen Pack| Stationery|  120|       3| 

## Task 3: Filtering, Grouping and Aggregation

In [3]:
# FILTERING: orders with sales above 1000 in the West or North region
print("Name: Vedant Vinod Rakshe")
df_sales.filter((col("sales") > 1000) & (col("region").isin("West", "North"))).show()

# GROUPING + AGGREGATION: sales summary per region
(df_sales.groupBy("region")
    .agg(count("order_id").alias("orders"),
         _sum("sales").alias("total_sales"),
         _round(avg("sales"), 2).alias("avg_sales"),
         _max("sales").alias("max_sale"))
    .orderBy(desc("total_sales"))
    .show())

Name: Vedant Vinod Rakshe
+--------+-----------+--------+-----------+-----+--------+------+
|order_id|customer_id| product|   category|sales|quantity|region|
+--------+-----------+--------+-----------+-----+--------+------+
|       1|        C01|  Laptop|Electronics|55000|       1|  West|
|       3|        C03|   Chair|  Furniture| 3500|       2| North|
|       4|        C01|Keyboard|Electronics| 1500|       1|  West|
|       6|        C05|    Desk|  Furniture| 9000|       1| North|
|       3|        C03|   Chair|  Furniture| 3500|       2| North|
+--------+-----------+--------+-----------+-----+--------+------+

+------+------+-----------+---------+--------+
|region|orders|total_sales|avg_sales|max_sale|
+------+------+-----------+---------+--------+
|  West|     4|      57120|  14280.0|   55000|
| South|     3|      25600|  8533.33|   25000|
| North|     3|      16000|  5333.33|    9000|
|  East|     2|      12080|   6040.0|   12000|
+------+------+-----------+---------+--------+


## Task 4: Remove Duplicate Records

In [4]:
# REMOVE DUPLICATE RECORDS
print("Name: Vedant Vinod Rakshe")
print("Rows before         :", df_sales.count())
df_distinct = df_sales.dropDuplicates()                      # identical rows
print("Rows after removing :", df_distinct.count())
print("Unique order_ids    :", df_sales.dropDuplicates(["order_id"]).count())
df_distinct.orderBy("order_id").show()

Name: Vedant Vinod Rakshe
Rows before         : 12
Rows after removing : 10
Unique order_ids    : 10
+--------+-----------+----------+-----------+-----+--------+------+
|order_id|customer_id|   product|   category|sales|quantity|region|
+--------+-----------+----------+-----------+-----+--------+------+
|       1|        C01|    Laptop|Electronics|55000|       1|  West|
|       2|        C02|     Mouse|Electronics|  500|       2|  West|
|       3|        C03|     Chair|  Furniture| 3500|       2| North|
|       4|        C01|  Keyboard|Electronics| 1500|       1|  West|
|       5|        C04|  Notebook| Stationery|  300|       5| South|
|       6|        C05|      Desk|  Furniture| 9000|       1| North|
|       7|        C02|  Pen Pack| Stationery|  120|       3|  West|
|       8|        C06|   Monitor|Electronics|12000|       1|  East|
|       9|        C07|      Sofa|  Furniture|25000|       1| South|
|      10|        C09|Pencil Box| Stationery|   80|       4|  East|
+--------+-----

## Task 5: Join Two Datasets

In [5]:
# JOIN the sales data with the customers data
print("Name: Vedant Vinod Rakshe")
joined_df = df_distinct.join(df_customers, on="customer_id", how="inner")
print("Inner join rows:", joined_df.count())
joined_df.select("order_id", "customer_name", "city", "product", "sales").orderBy("order_id").show()

# LEFT join keeps orders whose customer is missing in customers.csv (C09)
left_df = df_distinct.join(df_customers, on="customer_id", how="left")
left_df.filter(col("customer_name").isNull()).select("order_id", "customer_id", "product").show()

Name: Vedant Vinod Rakshe
Inner join rows: 9
+--------+-------------+------+--------+-----+
|order_id|customer_name|  city| product|sales|
+--------+-------------+------+--------+-----+
|       1|    Amit Shah|Mumbai|  Laptop|55000|
|       2|   Riya Patil|  Pune|   Mouse|  500|
|       3|     Neha Rao| Delhi|   Chair| 3500|
|       4|    Amit Shah|Mumbai|Keyboard| 1500|
|       5|   Karan Jain|Nashik|Notebook|  300|
|       6|    Sana Khan|  Pune|    Desk| 9000|
|       7|   Riya Patil|  Pune|Pen Pack|  120|
|       8|    Dev Mehta| Surat| Monitor|12000|
|       9|  Tina Dsouza|   Goa|    Sofa|25000|
+--------+-------------+------+--------+-----+

+--------+-----------+----------+
|order_id|customer_id|   product|
+--------+-----------+----------+
|      10|        C09|Pencil Box|
+--------+-----------+----------+


## Task 6: Calculate Average Sales by Product Category

In [6]:
# AVERAGE SALES BY PRODUCT CATEGORY
print("Name: Vedant Vinod Rakshe")
average_sales = (df_distinct.groupBy("category")
                 .agg(_round(avg("sales"), 2).alias("average_sales"))
                 .orderBy(desc("average_sales")))
average_sales.show()
spark.stop()

Name: Vedant Vinod Rakshe
+-----------+-------------+
|   category|average_sales|
+-----------+-------------+
|Electronics|      17250.0|
|  Furniture|      12500.0|
| Stationery|       166.67|
+-----------+-------------+


**Conclusion:** PySpark DataFrames were used to load CSV data with an inferred schema, filter and aggregate it, remove duplicate rows, join sales with customer details and compute the average sales for each product category. The same code scales from a local machine to a cluster because Spark distributes the work across partitions.